# EcoHome Energy Advisor: Database Setup

This notebook creates repeatable Berlin household data for energy use and rooftop solar generation.

In [1]:
from sqlalchemy import inspect
from models.energy import DatabaseManager, populate_berlin_sample_data
from tools import get_recent_energy_summary, query_energy_usage, query_solar_generation


In [2]:
db_manager = DatabaseManager()
counts = populate_berlin_sample_data(db_manager, days=30, seed=42)
print(f"Created {counts['usage_records']} energy records and {counts['solar_records']} solar records.")


Created 2160 energy records and 390 solar records.


In [3]:
table_names = set(inspect(db_manager.engine).get_table_names())
assert {'energy_usage', 'solar_generation'} <= table_names
print('Verified tables:', ', '.join(sorted(table_names)))


Verified tables: energy_usage, solar_generation


In [4]:
from datetime import datetime, timedelta
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=6)).strftime('%Y-%m-%d')
usage = query_energy_usage.invoke({'start_date': start_date, 'end_date': end_date})
solar = query_solar_generation.invoke({'start_date': start_date, 'end_date': end_date})
summary = get_recent_energy_summary.invoke({'hours': 24})
print('Seven day consumption:', usage['total_consumption_kwh'], 'kWh')
print('Seven day cost: EUR', usage['total_cost_eur'])
print('Seven day solar generation:', solar['total_generation_kwh'], 'kWh')
print('Recent device breakdown:', summary['usage']['device_breakdown'])


Seven day consumption: 578.72 kWh
Seven day cost: EUR 144.67
Seven day solar generation: 108.0 kWh
Recent device breakdown: {'EV': {'consumption_kwh': 47.49, 'cost_eur': 10.37, 'records': 24}, 'HVAC': {'consumption_kwh': 18.33, 'cost_eur': 5.77, 'records': 24}, 'appliance': {'consumption_kwh': 16.67, 'cost_eur': 5.19, 'records': 24}}


## What this means

The advisor now has a clean local history to compare device use, cost, and solar generation. The setup resets the demo data each time, so rerunning this notebook does not add duplicates.